|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>CUDA graphs<h1>|
|<h2>Lecture:</h2>|<h1><b>A graph refuses to change shape, and a server does nothing else<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# What a graph refuses to do

The capture froze everything: the pointers, the shapes and the control flow.
Replay runs the exact work that the capture recorded.

A server cannot accept that. Its batch size changes every step, which was the
whole point of continuous batching.

In [2]:
class Layer(nn.Module):
  """One transformer layer, shaped like a decode step: many small kernels."""
  def __init__(self, hidden):
    super().__init__()
    self.attention_norm = nn.RMSNorm(hidden)
    self.qkv_proj = nn.Linear(hidden, 3*hidden, bias=False)
    self.out_proj = nn.Linear(hidden, hidden, bias=False)
    self.mlp_norm = nn.RMSNorm(hidden)
    self.up_proj = nn.Linear(hidden, 4*hidden, bias=False)
    self.down_proj = nn.Linear(4*hidden, hidden, bias=False)

  def forward(self, hidden_states):
    normed = self.attention_norm(hidden_states)
    query, key, value = self.qkv_proj(normed).chunk(3, -1)
    scores = query @ key.transpose(-1, -2) / 32.0
    attention = torch.softmax(scores, -1) @ value
    hidden_states = hidden_states + self.out_proj(attention)
    normed = self.mlp_norm(hidden_states)
    return hidden_states + self.down_proj(F.silu(self.up_proj(normed)))

class Model(nn.Module):
  def __init__(self, num_layers=28, hidden=1024):
    super().__init__()
    self.layers = nn.ModuleList([Layer(hidden) for _ in range(num_layers)])

  def forward(self, hidden_states):
    for layer in self.layers:
      hidden_states = layer(hidden_states)
    return hidden_states

HIDDEN, LAYERS = 1024, 28

def random_input(num_seqs, hidden=HIDDEN):
  """The hidden states of one decode step: one token for each sequence."""
  return torch.randn(num_seqs, 1, hidden, device='cuda', dtype=torch.bfloat16)

In [3]:
def capture(model, example):
  """Record one forward pass as a graph.
  -> (graph, static_input, static_output).

  Warm up on a side stream first. cuBLAS allocates workspaces on the first
  call, and the graph must not record that allocation.
  """
  static_input = example.clone()
  side_stream = torch.cuda.Stream()
  side_stream.wait_stream(torch.cuda.current_stream())
  with torch.cuda.stream(side_stream):
    for _ in range(3):
      model(static_input)
  torch.cuda.current_stream().wait_stream(side_stream)
  graph = torch.cuda.CUDAGraph()
  with torch.cuda.graph(graph):
    static_output = model(static_input)
  return graph, static_input, static_output

In [4]:
model = Model(LAYERS, HIDDEN).cuda().to(torch.bfloat16).eval()
with torch.no_grad():
  graph, static_input, static_output = capture(model, random_input(4))
print(f'captured at batch {static_input.shape[0]}')

captured at batch 4


In [5]:
# Now a step arrives with 3 sequences, not 4.
try:
  static_input.copy_(random_input(3))
except RuntimeError as error:
  print('copy_ refused:', str(error).splitlines()[0])

copy_ refused: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 0


### The fix is the same one JAX needs, for a different reason

Capture a graph for each of a handful of **bucketed** batch sizes, and pad the
real batch up to the next [bucket](../../GLOSSARY.md#bucket). A step with 3 sequences runs the graph for
4, with one row of padding that produces a token nobody reads.

In [6]:
BUCKETS = [1, 2, 4, 8, 16, 32]

def bucket_for(num_seqs):
  """The smallest bucket that holds num_seqs, or None if no bucket does."""
  return next((bucket for bucket in BUCKETS if bucket >= num_seqs), None)

graphs = {}
with torch.no_grad():
  for bucket in BUCKETS:
    graphs[bucket] = capture(model, random_input(bucket))
print(f'captured {len(graphs)} graphs: {BUCKETS}')

for num_seqs in (1, 3, 5, 12, 31, 40):
  bucket = bucket_for(num_seqs)
  if bucket is None:
    print(f'{num_seqs:>3} sequences -> bucket None, no graph, run eager')
    continue
  padding = (bucket - num_seqs) / bucket
  print(f'{num_seqs:>3} sequences -> bucket {bucket:>4}, '
        f'{100*padding:4.0f}% of the rows are padding')

captured 6 graphs: [1, 2, 4, 8, 16, 32]
  1 sequences -> bucket    1,    0% of the rows are padding
  3 sequences -> bucket    4,   25% of the rows are padding
  5 sequences -> bucket    8,   38% of the rows are padding
 12 sequences -> bucket   16,   25% of the rows are padding
 31 sequences -> bucket   32,    3% of the rows are padding
 40 sequences -> bucket None, no graph, run eager


### And now measure whether it was worth it

In [7]:
def run_bucketed(num_seqs):
  graph, static_input, static_output = graphs[bucket_for(num_seqs)]
  static_input[:num_seqs].copy_(random_input(num_seqs))
  graph.replay()
  return static_output[:num_seqs]

print(f"{'seqs':>5} {'eager ms':>10} {'bucketed ms':>12} {'speedup':>8} {'padding':>8}")
for num_seqs in (1, 3, 5, 12, 31):
  step_input = random_input(num_seqs)
  with torch.no_grad():
    eager_ms = cudalib.bench_ms(lambda: model(step_input), iters=50, warmup=20, best_of=2)
  graph_ms = cudalib.bench_ms(lambda: run_bucketed(num_seqs), iters=50, warmup=20, best_of=2)
  bucket = bucket_for(num_seqs)
  print(f'{num_seqs:>5} {eager_ms:>10.3f} {graph_ms:>12.3f} {eager_ms/graph_ms:>7.2f}x '
        f'{100*(bucket-num_seqs)/bucket:>7.0f}%')

 seqs   eager ms  bucketed ms  speedup  padding


    1      4.826        2.582    1.87x       0%


    3      5.099        2.629    1.94x      25%


    5      5.060        2.633    1.92x      38%


   12      4.961        2.926    1.70x      25%


   31      4.975        2.787    1.79x       3%


### Two costs, and only one of them is obvious

**Padding.** A step with 5 sequences runs the graph for 8. The GPU then does
three rows of arithmetic that nobody uses. On the memory-bound left half of
the roofline those rows cost almost nothing. That is the only reason that this
method works.

**Memory.** Each captured graph holds its own input, output and intermediate
buffers for the life of the server. Six buckets means six copies.

The KV pool does not get that memory. So the bucket list trades against the
thing that Part 3 protected in four sections.

### The same fix, found two times

On the JAX track there are no kernel launches to remove. [XLA](../../GLOSSARY.md#xla) already [fused](../../GLOSSARY.md#fused-kernel) the
step into one graph. JAX has a different problem: **recompilation**. It costs
seconds, and it happens each time a new batch size appears.

The fix is the same. Pad to bucketed shapes, so that the compilation cache
hits.

The same answer comes from a completely different reason. That is the
strongest evidence that bucketing is not a CUDA trick. You do this whenever
the cost to prepare the work is large next to the work itself.

    ./vc guide 12